# Constrained generation (OpenRouter)

Same prompt, two runs:

| | Baseline | Constrained |
|---|---|---|
| messages | user only | system + strict user |
| stop | none | cut common postambles |
| expect | prose + code + extras | mostly just the code |

In [5]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(Path("/Users/garvitkhurana/Projects/llm-api-compare") / ".env")

API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert API_KEY, "Set OPENROUTER_API_KEY in .env"

URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

PROMPT = "Python code to give a sqrt of pi to 6 decimal places."
SYSTEM = "Be terse. Output only what was asked. No greetings or extras."
STOP = ["**Output", "Output:", "Here are", "Sure,"]

print("ok")

ok


In [6]:
def chat(messages, stop=None, temperature=0.2, max_tokens=600):
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if stop:
        payload["stop"] = stop

    r = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    data = r.json()
    if not r.ok or "error" in data or not data.get("choices"):
        raise RuntimeError(data.get("error") or data or r.text)

    choice = data["choices"][0]
    text = (choice.get("message") or {}).get("content") or ""
    return text, choice.get("finish_reason"), data.get("usage")

In [7]:
msgs_baseline = [{"role": "user", "content": PROMPT}]

msgs_constrained = [
    {"role": "system", "content": SYSTEM},
    {
        "role": "user",
        "content": PROMPT + "\nReply with only the Python code. No markdown fences, no explanation.",
    },
]

base_text, base_fr, base_usage = chat(msgs_baseline, temperature=0.7, max_tokens=800)
cons_text, cons_fr, cons_usage = chat(msgs_constrained, stop=STOP, temperature=0.2, max_tokens=400)

print("=" * 60)
print("BASELINE")
print("finish_reason:", base_fr, "| usage:", base_usage)
print("-" * 60)
print(base_text)

print("\n" + "=" * 60)
print("CONSTRAINED")
print("finish_reason:", cons_fr, "| usage:", cons_usage)
print("-" * 60)
print(cons_text)

RuntimeError: {'message': 'Upstream error from Nvidia: ResourceExhausted: Worker local total request limit reached (33/32)', 'code': 502}